In [1]:
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from glob import glob
from matplotlib.image import imread
from pathlib import Path

In [2]:
!mkdir results_breast/
!mkdir results_breast/breast_st_raw

In [3]:
for sample in ["1142243F", "1160920F", "CID4290", "CID4535", "CID4465", "CID44971"]:
    matrix_path = 'data/Wu2021st/filtered_count_matrices/'
    for f in glob(matrix_path + sample + "_filtered_count_matrix/*.gz"):
        g = f[0:-3]
        !mv $f $g
    adata = sc.read_mtx(matrix_path + sample + "_filtered_count_matrix/matrix.mtx").T
    with open(matrix_path + sample + "_filtered_count_matrix/barcodes.tsv", "r") as f:
        adata.obs_names = f.read().split("\n")[0:-1]
    with open(matrix_path + sample + "_filtered_count_matrix/features.tsv", "r") as f:
        adata.var_names = f.read().split("\n")[0:-1]

    adata.uns["spatial"] = dict()
    adata.uns["spatial"][sample] = dict()
    
    spatial_path = 'data/Wu2021st/spatial/{}_spatial/'.format(sample)
    files = dict(
        tissue_positions_file = spatial_path + 'tissue_positions_list.csv',
        scalefactors_json_file = spatial_path + 'scalefactors_json.json',
        hires_image = spatial_path + 'tissue_hires_image.png',
        lowres_image = spatial_path + 'tissue_lowres_image.png',)

    adata.uns["spatial"][sample]['images'] = dict()
    for res in ['hires', 'lowres']:
        adata.uns["spatial"][sample]['images'][res] = imread(str(files[f'{res}_image']))

    # read json scalefactors
    adata.uns["spatial"][sample]['scalefactors'] = json.loads(Path(files['scalefactors_json_file']).read_bytes())

    # read coordinates
    positions = pd.read_csv(files['tissue_positions_file'], header=None)
    positions.columns = ['barcode', 'in_tissue', 'array_row', 'array_col', 'pxl_col_in_fullres', 'pxl_row_in_fullres']
    positions.index = positions['barcode']

    adata.obs = adata.obs.join(positions, how="left")
    adata.obsm['spatial'] = adata.obs[['pxl_row_in_fullres', 'pxl_col_in_fullres']].to_numpy()
    adata.obs.drop(columns=['barcode', 'pxl_row_in_fullres', 'pxl_col_in_fullres'], inplace=True)
    
    metadata = pd.read_csv("data/Wu2021st/metadata/" + sample + "_metadata.csv", index_col=0)
    adata.obs["subtype"] = metadata["subtype"]
    adata.obs["pathology"] = metadata["Classification"]
    adata.obs["sample"] = sample
    adata.obs["replicate"] = sample
    adata.obs["ER"] = [x=="ER" for x in adata.obs["subtype"]]
    adata.obs["HER2"] = False # all samples are HER2 - in this dataset.
    adata.obs["PR"] = adata.obs["ER"] # All ER+ samples are also PR+ in this dataset.
    adata.layers["raw_counts"]=adata.X
    adata.obs_names = sample+"_"+adata.obs_names
    adata.write_h5ad("results_breast/breast_st_raw/{}.h5ad".format(sample))

In [4]:
sample_list = ["1142243F", "1160920F", "CID4290", "CID4535", "CID4465", "CID44971"]
adatas = dict()
for sample in sample_list:
    adatas[sample] = sc.read_h5ad("results_breast/breast_st_raw/{}.h5ad".format(sample))
adatas

{'1142243F': AnnData object with n_obs × n_vars = 4784 × 28402
     obs: 'in_tissue', 'array_row', 'array_col', 'subtype', 'pathology', 'sample', 'replicate', 'ER', 'HER2', 'PR'
     uns: 'spatial'
     obsm: 'spatial'
     layers: 'raw_counts',
 '1160920F': AnnData object with n_obs × n_vars = 4895 × 28402
     obs: 'in_tissue', 'array_row', 'array_col', 'subtype', 'pathology', 'sample', 'replicate', 'ER', 'HER2', 'PR'
     uns: 'spatial'
     obsm: 'spatial'
     layers: 'raw_counts',
 'CID4290': AnnData object with n_obs × n_vars = 2432 × 19237
     obs: 'in_tissue', 'array_row', 'array_col', 'subtype', 'pathology', 'sample', 'replicate', 'ER', 'HER2', 'PR'
     uns: 'spatial'
     obsm: 'spatial'
     layers: 'raw_counts',
 'CID4535': AnnData object with n_obs × n_vars = 1127 × 19237
     obs: 'in_tissue', 'array_row', 'array_col', 'subtype', 'pathology', 'sample', 'replicate', 'ER', 'HER2', 'PR'
     uns: 'spatial'
     obsm: 'spatial'
     layers: 'raw_counts',
 'CID4465': AnnDat

In [5]:
for sample in sample_list:
    print(adatas[sample].obs['pathology'])

1142243F_TACCGATCCAACACTT-1                                  Artefact
1142243F_GATAAGGGACGATTAG-1                                  Artefact
1142243F_TGTTGGCTGGCGGAAG-1                                  Artefact
1142243F_GCGAGGGACTGCTAGA-1                                  Artefact
1142243F_GCGCGTTTAAATCGTA-1                                  Artefact
                                                ...                  
1142243F_GAACGTTTGTATCCAC-1    Invasive cancer + stroma + lymphocytes
1142243F_ATTGAATTCCCTGTAG-1    Invasive cancer + stroma + lymphocytes
1142243F_TACCTCACCAATTGTA-1    Invasive cancer + stroma + lymphocytes
1142243F_AGTCGAATTAGCGTAA-1    Invasive cancer + stroma + lymphocytes
1142243F_TTGAAGTGCATCTACA-1    Invasive cancer + stroma + lymphocytes
Name: pathology, Length: 4784, dtype: category
Categories (6, object): ['Artefact', 'Invasive cancer + stroma + lymphocytes', 'Lymphocytes', 'Necrosis', 'Stroma', 'TLS']
1160920F_TGTTGGCTGGCGGAAG-1                            Adipo